# R4: Coordination Number Proxy vs RBF Distance — Causal Attribution

**Question**: Does RBF distance improve prediction via 3D stereochemistry, or merely as an implicit coordination-number encoder?

| Group | Model | Added Feature |
|-------|-------|---------------|
| Baseline | SMILES GAT | none (6d edge) |
| T1 | + CN one-hot | explicit coordination number (node feature) |
| T2 | + RBF 16 | RBF distances (edge features, 23d) |
| T3 | + CN + RBF | both |

If T2 > T1: supports unique value of 3D. If T1 ≈ T2: CN proxy explanation holds.
30k subsample, 1 seed each, ~30min per group.


In [1]:
import numpy as np, pandas as pd, time, gzip, os, json
import torch, torch.nn as nn, torch.optim as optim
from torch.amp import autocast, GradScaler
import warnings; warnings.filterwarnings("ignore")

os.environ.setdefault('OMP_NUM_THREADS', '16')
torch.set_num_threads(16)
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

from sklearn.metrics import r2_score
from tqdm import tqdm
from rdkit import Chem; from rdkit import RDLogger; RDLogger.logger().setLevel(RDLogger.ERROR)

df_y = pd.read_csv('tmQM_y.csv', sep=';')
df_y = df_y.rename(columns={'HL_Gap':'gap','HOMO_Energy':'homo','LUMO_Energy':'lumo','Dipole_M':'dipole','Metal_q':'metal_charge','Electronic_E':'E_elec','Dispersion_E':'E_disp','Polarizability':'polarizability'})
targets = ['gap','homo','lumo','dipole','metal_charge','E_elec','E_disp','polarizability']
print(f'Loaded {len(df_y)} complexes')


Device: cuda


Loaded 108541 complexes


In [2]:
# ===== Data: XYZ parse + 30k subsample (same as ablation) =====
def atom_to_z(sym):
    tbl = {'H':1,'He':2,'Li':3,'Be':4,'B':5,'C':6,'N':7,'O':8,'F':9,'Ne':10,'Na':11,'Mg':12,'Al':13,'Si':14,'P':15,'S':16,'Cl':17,'Ar':18,'K':19,'Ca':20,'Sc':21,'Ti':22,'V':23,'Cr':24,'Mn':25,'Fe':26,'Co':27,'Ni':28,'Cu':29,'Zn':30,'Br':35,'Ru':44,'Rh':45,'Pd':46,'Ag':47,'Cd':48,'I':53,'Pt':78,'Au':79,'Hg':80,'Mo':42,'W':74,'Ir':77,'Os':76,'Re':75}
    return tbl.get(sym,0)
def is_TM(z):
    return z in [21,22,23,24,25,26,27,28,29,30,39,40,41,42,43,44,45,46,47,48,72,73,74,75,76,77,78,79,80]

def parse_one(xyz_p, bo_p, start_idx=0):
    mols = []
    with gzip.open(bo_p, "rt") as f: bo_lines = [l.strip() for l in f.readlines()]
    with gzip.open(xyz_p, "rt") as f: xyz_blocks = f.read().strip().split(chr(10))
    bo_blocks, cur = [], []
    for line in bo_lines:
        if line.startswith("CSD_code"):
            if cur: bo_blocks.append(cur); cur = []
        elif line: cur.append(line)
    if cur: bo_blocks.append(cur)
    i, mol_idx = 0, start_idx - 1
    while i < len(xyz_blocks):
        line = xyz_blocks[i].strip(); i += 1
        if not line: continue
        parts = line.split()
        if len(parts)==1 and parts[0].isdigit():
            n_atoms = int(parts[0])
            if i >= len(xyz_blocks): break; i += 1
            atoms, coords = [], []
            for _ in range(n_atoms):
                if i >= len(xyz_blocks): break
                cline = xyz_blocks[i].strip(); i += 1; cparts = cline.split()
                if len(cparts)>=4:
                    try:
                        z = atom_to_z(cparts[0])
                        coords.append([float(x) for x in cparts[1:4]]); atoms.append(z)
                    except ValueError: continue
            mol_idx += 1
            if not any(is_TM(z) for z in atoms) or len(atoms)<=1: continue
            bo_mat = np.zeros((n_atoms,n_atoms),dtype=np.float32)
            if mol_idx < len(bo_blocks):
                for bline in bo_blocks[mol_idx]:
                    bparts = bline.split()
                    try: src_idx=int(bparts[0])-1
                    except ValueError: continue
                    j = 2
                    while j+1<len(bparts):
                        try:
                            nbr=int(bparts[j])-1; bo_val=float(bparts[j+1])
                            if 0<=src_idx<n_atoms and 0<=nbr<n_atoms: bo_mat[src_idx,nbr]=bo_val
                            j+=3
                        except (ValueError,IndexError): j+=1
            mols.append({"atoms":np.array(atoms),"coords":np.array(coords,dtype=np.float32),"bo":bo_mat,"csv_idx":mol_idx})
    return mols

print('Parsing XYZ+BO...')
all_m = []
for part in [1,2,3]:
    all_m.extend(parse_one(f"tmQM_X{part}.xyz.gz", f"tmQM_X{part}.BO.gz", start_idx=len(all_m)))
print(f'Total: {len(all_m)} TM')

# 30k subsample
rng = np.random.RandomState(42)
perm = rng.permutation(len(all_m))[:30000]
all_m = [all_m[i] for i in perm]
print(f'Subsampled 30k')

# SMILES graph constants
ELEMENT_DATA_SM = {1:(1,1),5:(2,13),6:(2,14),7:(2,15),8:(2,16),9:(2,17),14:(3,14),15:(3,15),16:(3,16),17:(3,17),20:(4,2),21:(4,3),22:(4,4),23:(4,5),24:(4,6),25:(4,7),26:(4,8),27:(4,9),28:(4,10),29:(4,11),30:(4,12),35:(4,17),42:(5,6),44:(5,8),45:(5,9),46:(5,10),47:(5,11),48:(5,12),53:(5,17),72:(6,4),73:(6,5),74:(6,6),75:(6,7),76:(6,8),77:(6,9),78:(6,10),79:(6,11),80:(6,12)}
BOND_TYPES_SM=[1,2,3,12]; B2I_SM={b:i for i,b in enumerate(BOND_TYPES_SM)}; N_BT_SM=4
rbf_c=np.linspace(1.0,5.0,16); rbf_sigma=0.2
smiles_by_idx=df_y["SMILES"].tolist()

def atom_features_sm(atom):
    z=atom.GetAtomicNum(); p,g=ELEMENT_DATA_SM.get(z,(0,0))
    return [z,atom.GetDegree(),atom.GetFormalCharge(),p,g,int(atom.GetIsAromatic()),atom.GetTotalNumHs(),int(z>=21)]

def fast_match(sm_atoms, xyz_atoms):
    N=len(sm_atoms)
    if N==len(xyz_atoms) and all(sm_atoms[i]==xyz_atoms[i] for i in range(N)): return list(range(N))
    sm_to_xyz=[-1]*N; used=set()
    for i,z_sm in enumerate(sm_atoms):
        for j,z_xyz in enumerate(xyz_atoms):
            if z_sm==z_xyz and j not in used: sm_to_xyz[i]=j; used.add(j); break
    return sm_to_xyz

def build_graph(m, use_cn=False, use_rbf=False):
    """Build graph with optional CN one-hot node feature and/or RBF edge features."""
    csv_i=m["csv_idx"]; smi=smiles_by_idx[csv_i] if csv_i<len(smiles_by_idx) else ""
    if not isinstance(smi,str) or smi=="": return None
    mol=Chem.MolFromSmiles(smi)
    if mol is None: return None
    N=mol.GetNumAtoms(); sm_atoms=[a.GetAtomicNum() for a in mol.GetAtoms()]
    sm_to_xyz=fast_match(sm_atoms, m["atoms"]); coords=m["coords"]; bo_mat=m.get("bo")

    # Node features: SMILES 8d (+ CN one-hot 8d if use_cn)
    xf=[atom_features_sm(a) for a in mol.GetAtoms()]
    if use_cn:
        for i in range(N):
            j=sm_to_xyz[i]
            if j>=0 and j<bo_mat.shape[0] and bo_mat is not None:
                cn=int((bo_mat[j]>0.05).sum())
            else:
                cn=0
            cn_onehot=[0.0]*8
            if 1<=cn<=8: cn_onehot[cn-1]=1.0
            xf[i].extend(cn_onehot)
    x=torch.tensor(xf,dtype=torch.float32)

    # Edge features: SMILES 6d (+ RBF 16d if use_rbf)
    src,dst,battr=[],[],[]
    for bond in mol.GetBonds():
        i,j=bond.GetBeginAtomIdx(),bond.GetEndAtomIdx(); bt=int(bond.GetBondType())
        za,zb=sm_atoms[i],sm_atoms[j]; is_ml=float((za>=21)!=(zb>=21)); is_ring=float(bond.IsInRing())
        src.extend([i,j]); dst.extend([j,i])
        bf=[0.0]*N_BT_SM
        if bt in B2I_SM: bf[B2I_SM[bt]]=1.0
        bf.extend([is_ring,is_ml])
        if use_rbf:
            d=2.0
            xi,xj=sm_to_xyz[i],sm_to_xyz[j]
            if xi>=0 and xj>=0 and xi<len(coords) and xj<len(coords):
                d=float(np.linalg.norm(coords[xi]-coords[xj]))
            for mu in rbf_c: bf.append(np.exp(-(d-mu)**2/rbf_sigma))
        battr.extend([bf,bf])
    ei=torch.tensor([src,dst],dtype=torch.long) if src else torch.zeros(2,0,dtype=torch.long)
    ea_dim=6+16 if use_rbf else 6
    ea=torch.tensor(battr,dtype=torch.float32) if battr else torch.zeros(0,ea_dim)
    return {"x":x,"edge_index":ei,"edge_attr":ea,"num_nodes":N}

print("Building 4 graph variants...")
graphs = {}
labels = {}
for variant in ["baseline", "cn", "rbf", "cn_rbf"]:
    use_cn = "cn" in variant
    use_rbf = "rbf" in variant
    gs, ys = [], []
    for m in tqdm(all_m, desc=f"Building {variant}"):
        g = build_graph(m, use_cn=use_cn, use_rbf=use_rbf)
        if g is None: continue
        gs.append(g)
        ys.append(df_y.iloc[m["csv_idx"]][targets].astype(float).values)
    y_all = torch.tensor(ys, dtype=torch.float32)
    mask = ~(torch.isnan(y_all).any(dim=1)|torch.isinf(y_all).any(dim=1))
    graphs[variant] = [g for i,g in enumerate(gs) if mask[i]]
    labels[variant] = y_all[mask]
    print(f"  {variant}: {len(graphs[variant]):,} graphs")


Parsing XYZ+BO...


Total: 96984 TM
Subsampled 30k


Building 4 graph variants...


Building baseline:   0%|          | 0/30000 [00:00<?, ?it/s]

Building baseline:   0%|          | 68/30000 [00:00<00:44, 677.31it/s]

...

Building cn_rbf: 100%|█████████▉| 29894/30000 [01:15<00:00, 427.15it/s]

Building cn_rbf: 100%|█████████▉| 29937/30000 [01:15<00:00, 412.43it/s]

Building cn_rbf: 100%|█████████▉| 29984/30000 [01:15<00:00, 428.13it/s]

Building cn_rbf: 100%|██████████| 30000/30000 [01:15<00:00, 398.53it/s]

  cn_rbf: 27,993 graphs


In [3]:
# ===== Shared model + training (identical for all 4 variants) =====
HIDDEN=384; N_LAYERS=5; GAT_HEADS=12; DROPOUT=0.10

def segment_softmax(scores, dst, N):
    scores_f32 = scores.float()
    max_v = torch.full((N, scores_f32.size(1)), float('-inf'), device=scores_f32.device, dtype=torch.float32)
    max_v.scatter_reduce_(0, dst.unsqueeze(-1).expand(-1, scores_f32.size(1)), scores_f32, reduce='amax', include_self=False)
    exp_s = torch.exp(scores_f32 - max_v[dst])
    sum_v = torch.zeros(N, scores_f32.size(1), device=scores_f32.device, dtype=torch.float32)
    sum_v.scatter_add_(0, dst.unsqueeze(-1).expand(-1, scores_f32.size(1)), exp_s)
    return (exp_s / (sum_v[dst] + 1e-8)).to(scores.dtype)

class GATConv(nn.Module):
    def __init__(self, in_dim, out_dim_per_head, edge_dim=0, heads=4):
        super().__init__(); self.heads, self.out_dim = heads, out_dim_per_head
        total_dim = out_dim_per_head * heads
        self.node_lin = nn.Linear(in_dim, total_dim, bias=False)
        self.attn_src = nn.Linear(total_dim, heads, bias=False)
        self.attn_dst = nn.Linear(total_dim, heads, bias=False)
        self.edge_attn = nn.Linear(edge_dim, heads, bias=False) if edge_dim>0 else None
        self.act = nn.ReLU()
    def forward(self, x, ei, ea=None):
        src, dst = ei[0], ei[1]; N = x.size(0); h = self.node_lin(x)
        attn = self.attn_src(h[src]) + self.attn_dst(h[dst])
        if self.edge_attn is not None and ea is not None and ea.size(0)>0: attn = attn + self.edge_attn(ea)
        attn = torch.nn.functional.leaky_relu(attn, 0.2)
        alpha = segment_softmax(attn, dst, N)
        h_heads = h.view(-1, self.heads, self.out_dim)
        msg = h_heads[src] * alpha.unsqueeze(-1); msg = msg.view(-1, self.out_dim * self.heads)
        out = torch.zeros(N, self.out_dim * self.heads, device=x.device, dtype=msg.dtype)
        out.scatter_add_(0, dst.unsqueeze(-1).expand_as(msg), msg)
        return self.act(out)

def collate_graphs(graph_list):
    x_list, ei_list, ea_list, batch_list = [], [], [], []
    offset = 0
    for i, g in enumerate(graph_list):
        n = g["x"].size(0)
        x_list.append(g["x"]); ei_list.append(g["edge_index"] + offset)
        ea_list.append(g["edge_attr"])
        batch_list.append(torch.full((n,), i, dtype=torch.long, device=g["x"].device))
        offset += n
    return {'x': torch.cat(x_list, dim=0), 'edge_index': torch.cat(ei_list, dim=1), 'edge_attr': torch.cat(ea_list, dim=0)}, torch.cat(batch_list, dim=0)

def scatter_mean_per_graph(h, batch, n):
    out = torch.zeros(n, h.size(1), device=h.device, dtype=h.dtype)
    out.scatter_add_(0, batch.unsqueeze(-1).expand_as(h), h)
    cnt = torch.zeros(n, device=h.device, dtype=h.dtype)
    cnt.scatter_add_(0, batch, torch.ones_like(batch, dtype=h.dtype))
    return out / cnt.clamp(min=1).unsqueeze(-1)

def scatter_max_per_graph(h, batch, n):
    out = torch.full((n, h.size(1)), float('-inf'), device=h.device, dtype=h.dtype)
    out.scatter_reduce_(0, batch.unsqueeze(-1).expand_as(h), h, reduce='amax', include_self=False)
    return out

class GAT_Model(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden=HIDDEN, n_layers=N_LAYERS, output_dim=8, dropout=DROPOUT, heads=GAT_HEADS):
        super().__init__(); self.heads = heads
        self.embed = nn.Linear(node_dim, hidden)
        self.convs = nn.ModuleList([GATConv(hidden, hidden//heads, edge_dim, heads=heads) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout)
        self.readout = nn.Sequential(
            nn.Linear(hidden*3, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden//2, output_dim))
    def forward(self, g, batch=None):
        x, ei, ea = g['x'], g['edge_index'], g.get('edge_attr', None)
        h = self.embed(x)
        for conv, norm in zip(self.convs, self.norms): h = norm(h + conv(h, ei, ea)); h = self.drop(h)
        if batch is None:
            mm = (x[:,7] > 0.5); mf = h[mm].mean(dim=0) if mm.any() else torch.zeros(h.size(1), device=h.device, dtype=h.dtype)
            return self.readout(torch.cat([h.mean(0), h.max(0).values, mf])).unsqueeze(0)
        n = int(batch.max().item()) + 1
        mean_h = scatter_mean_per_graph(h, batch, n); max_h = scatter_max_per_graph(h, batch, n)
        mm = (x[:,7] > 0.5); mf = scatter_mean_per_graph(h[mm], batch[mm], n) if mm.any() else torch.zeros(n, h.size(1), device=h.device, dtype=h.dtype)
        return self.readout(torch.cat([mean_h, max_h, mf], dim=1))

print("Model ready")


Model ready


In [4]:
# ===== Train all 4 variants (30k, 1 seed) =====
BATCH=128; EPOCHS=150; EARLY_STOP=40

def train_variant(variant):
    gs = graphs[variant]
    y_all = labels[variant]
    node_dim = gs[0]["x"].shape[1]
    edge_dim = gs[0]["edge_attr"].shape[1]

    torch.manual_seed(42); np.random.seed(42)
    model = GAT_Model(node_dim, edge_dim).to(device)
    opt = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=50, T_mult=2, eta_min=1e-6)
    crit = nn.MSELoss()
    scaler = GradScaler('cuda')

    y_mean = y_all.mean(dim=0); y_std = y_all.std(dim=0).clamp(min=1e-8)
    y_norm = (y_all - y_mean) / y_std

    idx = np.random.RandomState(42).permutation(len(gs))
    n_tr = int(len(gs)*0.7); n_va = int(len(gs)*0.15)
    tr_idx = idx[:n_tr]; va_idx = idx[n_tr:n_tr+n_va]; te_idx = idx[n_tr+n_va:]

    # GPU preload (30k fits)
    gs_gpu = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in g.items()} for g in tqdm(gs, desc='GPU upload')]
    yn_gpu = y_norm.to(device)

    best_va = float("inf"); best_state = None; patience = 0
    for ep in range(EPOCHS):
        model.train(); perm = torch.randperm(len(tr_idx))
        for s in range(0, len(tr_idx), BATCH):
            bidx = tr_idx[perm[s:s+BATCH]].tolist()
            bg = [gs_gpu[i] for i in bidx]; by = yn_gpu[bidx]
            batched, batch_idx = collate_graphs(bg)
            opt.zero_grad()
            with autocast('cuda'): loss = crit(model(batched, batch_idx), by)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        model.eval()
        with torch.no_grad():
            vp = []
            for s in range(0, len(va_idx), BATCH):
                bidx = va_idx[s:s+BATCH].tolist()
                bg = [gs_gpu[i] for i in bidx]
                batched, batch_idx = collate_graphs(bg)
                with autocast('cuda'): vp.append(model(batched, batch_idx).float())
            va_loss = nn.MSELoss()(torch.cat(vp), yn_gpu[va_idx])
        sched.step(ep + va_loss.item()/10)
        if va_loss < best_va:
            best_va = va_loss.item(); best_state = {k:v.clone() for k,v in model.state_dict().items()}; patience = 0
        else: patience += 1
        if patience >= EARLY_STOP:
            print(f'  {variant}: early stop @ ep {ep+1}'); break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        tp = []
        for s in range(0, len(te_idx), BATCH):
            bidx = te_idx[s:s+BATCH].tolist()
            bg = [gs_gpu[i] for i in bidx]
            batched, batch_idx = collate_graphs(bg)
            with autocast('cuda'): tp.append(model(batched, batch_idx).float().cpu())
        tp = torch.cat(tp)
    tp_orig = tp * y_std + y_mean; tt = y_all[te_idx]
    r2s = [r2_score(tt[:,i].numpy(), tp_orig[:,i].numpy()) for i in range(8)]
    mean_r2 = np.mean(r2s)
    print(f'  {variant}: Mean R2 = {mean_r2:.4f}, dipole R2 = {r2s[3]:.4f}')
    return {"variant": variant, "mean_r2": mean_r2, "r2s": r2s, "node_dim": node_dim, "edge_dim": edge_dim}

results = {}
for variant in ["baseline", "cn", "rbf", "cn_rbf"]:
    print(f'\n=== Training {variant} ===')
    results[variant] = train_variant(variant)

with open('cn_control_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved cn_control_results.json')



=== Training baseline ===


GPU upload:   0%|          | 0/27993 [00:00<?, ?it/s]

GPU upload:   7%|▋         | 1959/27993 [00:00<00:01, 19584.79it/s]

GPU upload:  14%|█▍        | 3918/27993 [00:00<00:05, 4485.51it/s] 

GPU upload:  22%|██▏       | 6226/27993 [00:00<00:02, 7485.88it/s]

...

GPU upload:  83%|████████▎ | 23292/27993 [00:01<00:00, 21428.51it/s]

GPU upload:  91%|█████████ | 25435/27993 [00:01<00:00, 20078.13it/s]

GPU upload:  98%|█████████▊| 27460/27993 [00:01<00:00, 18854.93it/s]

GPU upload: 100%|██████████| 27993/27993 [00:01<00:00, 20272.61it/s]

  cn_rbf: Mean R2 = 0.8491, dipole R2 = 0.4679

Saved cn_control_results.json


## Interpretation

| Variant | Mean R2 | Dipole R2 |
|----------|---------|-----------|
| baseline (SMILES 6d) | ? | ? |
| CN one-hot | ? | ? |
| RBF 16 | ? | ? |
| CN + RBF | ? | ? |

- If RBF (T2) > CN (T1): RBF distances provide information beyond coordination number → supports 3D stereochemistry explanation
- If CN (T1) ≈ RBF (T2): coordination-number proxy explanation holds → need to temper 3D claims
